This notebook attempts to detemine what variables require weighting, and how to apply the weighting.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import pygmt
from pygmt_helper import plotting

import nn_gmm as nng

In [ ]:
imdb_ffp = Path("/Users/claudy/dev/work/data/nn_gmm/20250606_CS200m_imdb.db")
test_events_ffps = Path("/Users/claudy/dev/work/data/nn_gmm/test_events.npy")

In [ ]:
with nng.IMDB(imdb_ffp, readonly=True) as imdb:
    site_df = imdb.get_site_df(max_grid_level=0, min_grid_level=0)
    sites = site_df.site_id.values.astype(str)

    event_df = imdb.get_event_df()
    # Drop test events
    test_events = np.load(test_events_ffps).astype(str)
    event_df = event_df[~event_df.event_id.isin(test_events)]

    record_info_df = imdb.get_record_info_df(sites=sites, events=event_df.event_id.values.astype(str))
    site_event_data = imdb.get_site_event_df(sites=site_df.site_id.values.astype(str))
    
print("Event DF shape:", event_df.shape)

In [ ]:
# Add site-event int id
record_info_df["site_event_int_id"] = nng.utils.get_site_event_int_id(record_info_df.site_int_id.values, record_info_df.event_int_id.values)

# Add magnitude, rrup and vs30 data
record_info_df["vs30"] = site_df.loc[record_info_df.site_int_id, "vs30"].values
record_info_df["magnitude"] = event_df.loc[record_info_df.event_int_id, "magnitude"].values
record_info_df["rrup"] = site_event_data.loc[record_info_df.site_event_int_id, "rrup"].values

## Magnitude

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

ax.hist(record_info_df.magnitude, bins=30, edgecolor="black")
ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax.xaxis.set_minor_locator(plt.MultipleLocator(0.25))
ax.set_xlabel("Magnitude, $M_W$")
ax.set_ylabel("Number of records")
ax.set_title("Magnitude Distribution")
fig.tight_layout()

In [ ]:
mag_bins = nng.constants.MAG_WEIGHTING_BINS
mag_bin_names = nng.constants.MAG_WEIGHTING_BIN_NAMES

# mag_bins = np.array([5.5, 6.5, 7.25, 8.5])
# mag_bin_names = [
    # f"{mag_bins[i]:.2f}_{mag_bins[i + 1]:.2f}"
    # for i in range(len(mag_bins) - 1)
# ]

record_info_df["mag_bin"] = pd.cut(record_info_df.magnitude, mag_bins, labels=mag_bin_names)

mag_bin_counts = record_info_df.mag_bin.value_counts().sort_index()

In [ ]:
# Note: Additional weight due to magnitude, base weight is 1
mag_max_weight = 2.5
mag_bin_weights = np.clip(mag_bin_counts.max() / mag_bin_counts, 1.0, mag_max_weight) - 1
record_info_df["mag_bin_weight"] = record_info_df.mag_bin.map(mag_bin_weights).astype(float)
print("Using max mag weight:", mag_max_weight)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.hist(record_info_df.magnitude, bins=mag_bins, edgecolor="black")
ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.xaxis.set_minor_locator(plt.MultipleLocator(0.25))
ax1.set_title("Original")
ax1.set_ylabel("Count")
ax1.set_xlabel("Magnitude Bin")

ax2.hist(record_info_df.magnitude, bins=mag_bins, weights=1 + record_info_df["mag_bin_weight"], edgecolor="black")
ax2.set_title("Weighted")
ax2.set_xlabel("Magnitude Bin")
ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax2.xaxis.set_minor_locator(plt.MultipleLocator(0.25))

fig.tight_layout()

## $R_{Rup}$

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.hist(record_info_df.rrup, bins=10, edgecolor="black")
ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.xaxis.set_minor_locator(plt.MultipleLocator(25))
ax1.set_title("$R_{Rup}$ Distribution")

log_rrup_bins = np.logspace(np.log(1), np.log(300), 10, base=np.e)
ax2.hist(record_info_df.rrup, bins=log_rrup_bins, edgecolor="black")
ax2.set_title("$R_{Rup}$ Distribution (Log Scale)")
ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax2.set_xscale("log")
# ax2.xaxis.set_minor_locator(plt.MultipleLocator(25))

fig.tight_layout()

In [ ]:
rrup_bins = nng.constants.RRUP_WEIGHTING_BINS
rrup_bin_names = nng.constants.RRUP_WEIGHTING_BIN_NAMES

# rrup_bins = np.asarray([0, 30, 100, 300])
# rrup_bin_names = [
    # f"{rrup_bins[i]:.2f}_{rrup_bins[i + 1]:.2f}"
    # for i in range(len(rrup_bins) - 1)
# ]

record_info_df["rrup_bin"] = pd.cut(record_info_df.rrup, rrup_bins, labels=rrup_bin_names)

rrup_bin_counts = record_info_df.rrup_bin.value_counts().sort_index()

In [ ]:
# Note: Additional weight due to rrup, base weight is 1
max_rrup_weight = 2.5
rrup_bin_weights = np.clip(rrup_bin_counts.max() / rrup_bin_counts, 1.0, max_rrup_weight) - 1
record_info_df["rrup_bin_weight"] = record_info_df.rrup_bin.map(rrup_bin_weights).astype(float)
print("Using max rrup weight:", max_rrup_weight)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.hist(record_info_df.rrup, bins=rrup_bins, edgecolor="black")
ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.xaxis.set_minor_locator(plt.MultipleLocator(25))
ax1.set_title("Original")
ax1.set_ylabel("Count")
ax1.set_xlabel("Rrup Bin")

ax2.hist(record_info_df.rrup, bins=rrup_bins, weights=1 + record_info_df["rrup_bin_weight"], edgecolor="black")
ax2.set_title("Weighted")
ax2.set_xlabel("Rrup Bin")
ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
# ax2.xaxis.set_minor_locator(plt.MultipleLocator(25))

fig.tight_layout()

## $V_{S30}$

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.hist(record_info_df.vs30, bins=10, edgecolor="black")
ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.xaxis.set_minor_locator(plt.MultipleLocator(25))
ax1.set_title("$Vs30$ Distribution")

log_vs30_bins = nng.constants.VS30_WEIGHTING_BINS
ax2.hist(record_info_df.vs30, bins=log_vs30_bins, edgecolor="black")
ax2.set_title("$Vs30$ Distribution (Log Scale)")
ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax2.xaxis.set_minor_locator(plt.MultipleLocator(25))
# ax2.set_xscale("log")

fig.tight_layout()

This one is the trickiest as there there is significant more sites in the 400-500 range than any other Vs30 values.

In [ ]:
vs30_bins = nng.constants.VS30_WEIGHTING_BINS
vs30_bin_names = nng.constants.VS30_WEIGHTING_BIN_NAMES

# vs30_bins = [0, 180, 360, 760, 1500]
# vs30_bin_names = [
    # f"{vs30_bins[i]:.2f}_{vs30_bins[i + 1]:.2f}"
    # for i in range(len(vs30_bins) - 1)
# ]

record_info_df["vs30_bin"] = pd.cut(record_info_df.vs30, vs30_bins, labels=vs30_bin_names)

vs30_bin_counts = record_info_df.vs30_bin.value_counts().sort_index()

In [ ]:
# Note: Additional weight due to Vs30, base weight is 1
max_vs30_weight = 2.5
vs30_bin_weights = np.clip(vs30_bin_counts.max() / vs30_bin_counts, 1.0, max_vs30_weight) - 1
record_info_df["vs30_bin_weight"] = record_info_df.vs30_bin.map(vs30_bin_weights)
print("Using max vs30 weight:", max_vs30_weight)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.hist(record_info_df.vs30, bins=vs30_bins, edgecolor="black")
ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.xaxis.set_minor_locator(plt.MultipleLocator(25))
ax1.set_title("Original")
ax1.set_ylabel("Count")
ax1.set_xlabel("Vs30 Bin")

ax2.hist(record_info_df.vs30, bins=vs30_bins, weights=1 + record_info_df["vs30_bin_weight"], edgecolor="black")
ax2.set_title("Weighted")
ax2.set_xlabel("Vs30 Bin")
ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax2.xaxis.set_minor_locator(plt.MultipleLocator(25))

fig.tight_layout()